In [67]:
import datasets
print(datasets.__version__)


4.8.5


### Loading Dataset

In [77]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

# take small subset
texts = dataset["train"]["text"][:10000]

In [69]:
print(type(texts))
print(len(texts))
print(texts[:3])

<class 'list'>
10000
['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.', 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had

In [78]:
import os
import pickle

save_dir = "/home/mohd_kaif/minigpt/data/raw_dataset"

file_path = os.path.join(save_dir, "texts_10k.pkl")

with open(file_path, "wb") as f:
    pickle.dump(texts, f)

print("Saved at:", file_path)

Saved at: /home/mohd_kaif/minigpt/data/raw_dataset/texts_10k.pkl


### Cleaning

In [60]:
with open("/home/mohd_kaif/minigpt/data/raw_dataset/texts_10k.pkl", "rb") as f:
    texts = pickle.load(f)

In [79]:
import re

def clean_text(text):
    text = text.lower()
    text = text.replace("\n", " ")
    
    # keep useful punctuation
    text = re.sub(r"[^a-z0-9.,!?\'\" ]", "", text)
    
    # normalize spaces
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

cleaned_texts = [clean_text(t) for t in texts]

print("Cleaned samples:", len(cleaned_texts))

Cleaned samples: 10000


In [74]:
def split_into_chunks(texts, max_len=64):
    all_chunks = []

    for text in texts:
        words = text.split()

        for i in range(0, len(words), max_len):
            chunk = words[i:i+max_len]

            if len(chunk) >= 5:   # ignore very small chunks
                all_chunks.append(" ".join(chunk))

    return all_chunks


chunked_texts = split_into_chunks(cleaned_texts, max_len=64)

print("Total chunks:", len(chunked_texts))

Total chunks: 30788


In [80]:
final_texts = ["<start> " + t + " <end>" for t in chunked_texts]

print(final_texts[:3])

['<start> one day, a little girl named lily found a needle in her room. she knew it was difficult to play with it because it was sharp. lily wanted to share the needle with her mom, so she could sew a button on her shirt. lily went to her mom and said, "mom, i found this needle. can you share it with me and sew <end>', '<start> my shirt?" her mom smiled and said, "yes, lily, we can share the needle and fix your shirt." together, they shared the needle and sewed the button on lily\'s shirt. it was not difficult for them because they were sharing and helping each other. after they finished, lily thanked her mom for sharing the needle and fixing her shirt. they both felt happy because <end>', '<start> they had shared and worked together. <end>']


In [81]:
save_path = "/home/mohd_kaif/minigpt/data/processed_dataset"
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, "final_texts.pkl"), "wb") as f:
    pickle.dump(final_texts, f)

print("Saved processed data")

Saved processed data
